# 03 - Inference
Loads the packaged pipeline (`models/model.joblib`) exactly as the API does and scores a few patients. The second half calls the running API.

In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
from IPython.display import Image, display
pd.set_option("display.width", 140)

In [ ]:
import json
from heartrisk.predict import RiskModel
model = RiskModel.load()
print(model.meta['model_name'], model.version, 'threshold', model.threshold)

In [ ]:
patients = [json.load(open('samples/high_risk.json')), json.load(open('samples/low_risk.json'))]
pd.DataFrame(model.predict(patients))

Missing `ca` / `thal` are allowed - the pipeline imputes them.

In [ ]:
p = dict(patients[0], ca=None, thal=None)
model.predict([p])

## Call the API (start it first: `uvicorn heartrisk.api:app --port 8000` or the Docker container)

In [ ]:
import urllib.request
def call(url, payload=None):
    req = urllib.request.Request(url, data=json.dumps(payload).encode() if payload else None,
                                 headers={'Content-Type': 'application/json'})
    with urllib.request.urlopen(req, timeout=5) as r:
        return json.loads(r.read())
try:
    print(call('http://localhost:8000/predict', patients[0]))
except Exception as e:
    print('API not running:', e)